# Imports

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

  
if Path.cwd().name in ['ETL', 'data_science']:
    root = Path.cwd().parent
else:
    root = Path.cwd()

if str(root) not in sys.path:
    sys.path.append(str(root))
else:
    None
from utils import align_working_dir, save_chart

align_working_dir('data_science') 

%run ./0_content_load.ipynb

In [ ]:
# Save charts to a specific folder with a given prefix in the name.
CHART_PREFIX = "all_"

# Annual Activity Comparison
- **Clustered Bar Chart**  

[NOTE]  
  - **X-axis**: Years (e.g., 2023, 2024, 2025).
  - **Y-axis**: Total numbers for articles / blog posts / events.
  - **Details**: For each year, a separate column for articles, blog post and events.

In [ ]:
# Group by Year and Type
annual_activity_df = combined_df.groupby(['Year', 'Type']).size().reset_index(name='Count')

display(annual_activity_df.head(2))

sum_articles = annual_activity_df[annual_activity_df['Type'] == 'Article']['Count'].sum()
sum_blog_posts = annual_activity_df[annual_activity_df['Type'] == 'Blog Post']['Count'].sum()
sum_events = annual_activity_df[annual_activity_df['Type'] == 'Event']['Count'].sum()

# --- Plotting ---
fig, ax = plt.subplots(figsize=(10, 6))

sns.barplot(
    data=annual_activity_df, 
    x='Year', 
    y='Count', 
    hue='Type', 
    palette=CUSTOM_PALETTE,
    hue_order=['Article', 'Blog Post', 'Event'],
    ax=ax
)

# Add labels on top of the bars
for container in ax.containers: ax.bar_label(container, padding=3)

fig.suptitle('Annual Activity Comparison') 
ax.set_title(f'Distribution of content types over recorded period', linespacing=1.8)

ax.set_ylabel('Total Count')
ax.set_xlabel('Year')

# Ensure ticks (both x and y) appear as per global style
ax.tick_params(axis='both', which='major', left=True, bottom=True)

# Get the handles (the colored bars) created by Seaborn
handles, _ = ax.get_legend_handles_labels()

ax.legend(
    handles=handles, 
    labels=[f'Article (n={sum_articles})', f'Blog Post (n={sum_blog_posts})', f'Event (n={sum_events})'],
    title='Content Type:',
    loc='upper right', # The 'anchor point' of the legend box itself
    alignment='left'
)

# Bypass "Unsupported MimeType" error in VS Code Jupyter Notebooks when using Docker
save_chart(fig, "annual_activity_comparison", CHART_PREFIX)
plt.show()

# Monthly Activity Comparison
- **Line Chart**  

[NOTE]   
Monthly Seasonality (Monthly Trends)  
  - **X-axis**: Months (1-12) in a given year.
  - **Y-axis**: Total numbers for articles / blog posts / events.
  - **Details**: Three lines on the same chart for articles, blog post and events.

In [ ]:
# Group by Month and Type (aggregating all years for general seasonality)
monthly_trends_df = combined_df.groupby(['Month', 'Type']).size().reset_index(name='Count')

display(monthly_trends_df.head(2))

# --- Plotting ---
fig, ax = plt.subplots(figsize=(12, 7))

month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep',
   'Oct', 'Nov', 'Dec']

sns.lineplot(
    data=monthly_trends_df,
    x='Month',
    y='Count',
    hue='Type',
    palette=CUSTOM_PALETTE,
    hue_order=['Article', 'Blog Post', 'Event'],
    marker='o',
    markersize=8,
    linewidth=1,
    ax=ax
)

fig.suptitle('Monthly Activity Comparison')
ax.set_title(f'Distribution of content types over recorded period\n(2019 - 2025)\n Total Articles: {sum_articles} | Total Blog Posts: {sum_blog_posts} | Total Events: {sum_events}', linespacing=1.8)

ax.set_ylabel('Total Count')
ax.set_xlabel('Month')

ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_labels)

# Ensure ticks (both x and y) appear as per global style
ax.tick_params(axis='both', which='major', left=True, bottom=True)

ax.legend(
    title='Content Type:',
    bbox_to_anchor=(1, 1.15),
    loc='upper right', 
)

# Bypass "Unsupported MimeType" error in VS Code Jupyter Notebooks when using Docker
save_chart(fig, "monthly_activity_comparison", CHART_PREFIX)
plt.show()

# Monthly Activity Comparison
-  **Scatterplot**  

[NOTE]   
Monthly Seasonality (Monthly Trends)  

   * **X-axis**: Months (1–12).
   * **Y-axis**: Items published in that specific month/year.
   * **Dots**: Each dot represents a single record (article, blog post or event) published in a given month of a given year.

In [ ]:
# Ensure the column exists before plotting
combined_df['Year_Str'] = combined_df['Year'].astype(str)

# --- Plotting ---
fig, ax = plt.subplots(figsize=(15, 15)) 

sns.swarmplot(
    data=combined_df, 
    x='Month', 
    y='Year_Str',
    hue='Type', 
    palette=CUSTOM_PALETTE,
    hue_order=['Article', 'Blog Post', 'Event'],
    size=5,        
    dodge=True,    
    ax=ax
)

plt.xticks(ticks=range(1, 13), labels=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])

years = sorted(combined_df['Year_Str'].unique(), reverse=True) # Sorted for logical order
ax.set_yticks(range(len(years)))
ax.set_yticklabels(years, va='center') 

fig.suptitle('Content Publication Timeline (2019 - 2025)') 
ax.set_title('Each dot represents one item | Grouped by Year and Month')

ax.set_xlabel('Month')
ax.set_ylabel('Year')

# Ensure ticks (both x and y) appear as per global style
ax.tick_params(axis='both', which='major', left=True, bottom=True)

ax.legend(
    title='Content Type:',
    bbox_to_anchor=(1, 1),
    loc='upper right', # The 'anchor point' of the legend box itself
)

plt.show()

  # Yearly Strategy Shift: Content Mix Ratio  
  **Stacked Bar Chart**  

  [NOTE]  
  Instead of just showing the counts, show the ratio.
 * **Visual**: 100% Stacked Bar Chart.
 * **Insight**: E.g., "In 2021, 80% of our output was Blog Posts; by 2025, Events represent 60% of our activity."

In [ ]:
# Aggregate: Get counts by Year and Type
yearly_content_mix_df = (
    combined_df.groupby(['Year', 'Type'])
    .size()
    .unstack(fill_value=0) # Converts 'Type' from rows to columns
)

display(yearly_content_mix_df.head(2))

# Normalize: Convert raw counts to percentages (0 to 100%)
# We divide each row by its total sum across all columns
yearly_mix_pct_df = (
    yearly_content_mix_df
    .div(yearly_content_mix_df.sum(axis=1), axis=0) * 100
)

# --- Plotting ---
ax=yearly_mix_pct_df.plot(
    kind='bar',
    stacked=True,
    figsize=(12, 8),
    color=[CUSTOM_PALETTE[col] for col in yearly_mix_pct_df.columns], 
    width=0.8,
    rot=0
)

fig = ax.get_figure()

fig.suptitle('Annual Content Mix Ratio')

ax.set_ylabel('Percentage of Total Output (%)')
ax.set_xlabel('Year')

# Ensure ticks (both x and y) appear as per global style
ax.tick_params(axis='both', which='major', left=True, bottom=True)

# Get handles to keep colors, then update labels with totals
handles, _ = ax.get_legend_handles_labels()
ax.legend(
       handles=handles,
       labels=[f'Article (n={sum_articles})', f'Blog Post (n={sum_blog_posts})',
   f'Event (n={sum_events})'],
       loc='lower center',
       bbox_to_anchor=(0.5, 1.05),       
       ncol=3,
       frameon=False, 
   )

# Add percentage labels inside the bars for clarity
for p in ax.patches:
    width, height = p.get_width(), p.get_height()
    if height > 5: # Only label bars large enough to fit text
        x, y = p.get_xy()
        ax.annotate(f'{height:.0f}%',
                    (x + width/2, y + height/2),
                    ha='center', va='center',
                    fontsize=16, color='white', fontweight='medium')
        
# Bypass "Unsupported MimeType" error in VS Code Jupyter Notebooks when using Docker
save_chart(fig, "annual_strategy_shift", CHART_PREFIX)
plt.show()

# T# Annual Content Activity Pulse
- **Heatmap**  

  When is the team actually working most intensely?
   * **Visual**: A Heatmap with Months on the X-axis and Years on the Y-axis.
   * **Logic**: `pivot_table` of counts.
   * **Insight**: E.g., "There are "dead zones" in August and "peak seasons" in October."

In [ ]:
# Aggregate Count all content items by Year and Month
activity_heatmap_df = (
    combined_df.groupby(['Year', 'Month'])
    .size()
    .reset_index(name='Content_Count')
)
# Pivot Reshape data for the heatmap (Rows = Year, Columns = Month)
# Fill NaN with 0 because a missing month means 0 activity
activity_matrix_df = (
    activity_heatmap_df.pivot(index='Year', columns='Month', values='Content_Count')
    .fillna(0)
    .sort_index(ascending=False) 
    .astype(int)
)

month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep',
   'Oct', 'Nov', 'Dec']

# --- Plotting ---
fig, ax = plt.subplots(figsize=(10, 6))

ax = sns.heatmap(
    activity_matrix_df,
    annot=True,        
    fmt='d',           
    cmap='PuRd',
    xticklabels=month_labels,       
    cbar_kws={'label': 'Number of Items Published'}
)

# Manually turn on ONLY the left and bottom spines
ax.spines['left'].set_visible(True)
ax.spines['bottom'].set_visible(True)

for spine in ['left', 'bottom']:
    ax.spines[spine].set_color('#292929') 
    ax.spines[spine].set_linewidth(1.0)

# Ensure ticks (both x and y) appear as per global style
ax.tick_params(axis='both', which='major', left=True, bottom=True)

fig.suptitle('Annual Content Activity Pulse', x=0.08, y=0.98, ha='left')
ax.set_title(f'Frequency of content released per month across the years \nArticles n={sum_articles} | Blog Posts n={sum_blog_posts} | Events n={sum_events}', loc='left', pad=20)


ax.set_xlabel('Month of Year')
ax.set_ylabel('Year')

# Bypass "Unsupported MimeType" error in VS Code Jupyter Notebooks when using Docker
save_chart(fig, "annual_content_activity_heatmap", CHART_PREFIX)
plt.show()

# Cross-Type "Engagement Efficiency" : Blogs vs. Events
  Even though "Views" (Blog) and "Going" (Events) are different metrics, you can look at the **Average Engagement per Post** by Year.
  * **Logic**:
       * **Avg_Blog_Views = Total_Views / Post_Count**
       * **Avg_Event_Attendance = Total_Going / Event_Count**
  * **Insight**: E.g., "While we produce fewer events than blog posts, each event engages 5x more people on average than a single blog post."

In [ ]:
# Calculate Annual Blog Efficiency
annual_blog_efficiency_df = (
    combined_df.query("Type == 'Blog Post'")
    .groupby('Year')
    .agg(
        Total_Views=('Blog_Views', 'sum'),
        Post_Count=('Type', 'count')
    )
    .assign(Avg_Views_per_Post=lambda x: x['Total_Views'] / x['Post_Count'])
    .reset_index()
)

# Calculate Annual Event Efficiency
annual_event_efficiency_df = (
    combined_df.query("Type == 'Event'")
    .groupby('Year')
    .agg(
        Total_Attendance=('Guest_Going', 'sum'),
        Event_Count=('Type', 'count')
    )
    .assign(Avg_Attendance_per_Event=lambda x: x['Total_Attendance'] / x['Event_Count'])
    .reset_index()
)

annual_blog_efficiency_df = annual_blog_efficiency_df.copy()
annual_event_efficiency_df = annual_event_efficiency_df.copy()

annual_blog_efficiency_df['Metric_Type'] = 'Blog (Avg Views)'
annual_blog_efficiency_df['Avg_Engagement'] = annual_blog_efficiency_df['Avg_Views_per_Post']

annual_event_efficiency_df['Metric_Type'] = 'Event (Avg Attendance)'
annual_event_efficiency_df['Avg_Engagement'] = annual_event_efficiency_df['Avg_Attendance_per_Event']

# Combine into a Tidy Format for plotting
# 'annual_efficiency_summary_df'
annual_efficiency_summary_df = (
    pd.concat([annual_blog_efficiency_df, annual_event_efficiency_df])
    .reset_index()
)

- **Clustered Bar Chart** 

In [ ]:
# --- Plotting ---
fig, ax = plt.subplots(figsize=(10, 6))

sns.barplot(
    data=annual_efficiency_summary_df,
    x='Year',
    y='Avg_Engagement',
    hue='Metric_Type',
    palette={'Blog (Avg Views)': CUSTOM_PALETTE['Blog Post'], 'Event (Avg Attendance)': CUSTOM_PALETTE['Event']} 
)

# Ensure ticks (both x and y) appear as per global style
ax.tick_params(axis='both', which='major', left=True, bottom=True)

fig.suptitle('Engagement Efficiency: Blogs vs. Events',)
ax.set_ylabel('Average Engagement (Views or Attendees)')
ax.set_xlabel('Year')

# Add values on top of bars for precision
for p in plt.gca().patches:
    if p.get_height() > 0:
        plt.gca().annotate(f'{p.get_height():.1f}',
                           (p.get_x() + p.get_width() / 2., p.get_height()),
                           ha='center', va='center', fontsize=10,
                           color='black', xytext=(0, 7),
                           textcoords='offset points')

plt.show()

- **Dual-Axis Plotting**

In [ ]:
# --- Plotting ---
fig, ax1 = plt.subplots(figsize=(12, 7))

ax=sns.barplot(
    data=annual_efficiency_summary_df.query("Metric_Type == 'Blog (Avg Views)'"),
    x='Year', y='Avg_Engagement', ax=ax1, color=CUSTOM_PALETTE['Blog Post'], label='Blog Views', legend=False
)
ax1.tick_params(axis='x', bottom=True)
ax1.set_ylabel('Avg Blog Views', color=CUSTOM_PALETTE['Blog Post'])
ax1.tick_params(axis='y')

# Create second axis and Plot Event Efficiency (Line)
ax2 = ax1.twinx()

# Ensure the right spine is visible
ax2.spines['right'].set_visible(True)

sns.lineplot(
    data=annual_efficiency_summary_df.query("Metric_Type == 'Event (Avg Attendance)'"),
    x=range(len(annual_efficiency_summary_df['Year'].unique())), 
    y='Avg_Engagement', ax=ax2, color=CUSTOM_PALETTE['Event'], marker='o', linewidth=3, markersize=10,
    label='Event Attendance', legend=False
)

ax2.set_ylabel('Avg Event Attendance', color=CUSTOM_PALETTE['Event'])
ax2.tick_params(axis='y')

# Gather handles and labels from both axes
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()

# bbox_to_anchor=(0.5, 1.1) moves it to the center (0.5) and above the chart (1.1)
ax1.legend(h1 + h2, l1 + l2, 
           loc='lower center', 
           bbox_to_anchor=(0.5, 1.02), 
           ncol=2, 
           frameon=False, 
)

fig.suptitle('Engagement Efficiency: Blog vs Event', y=0.98)

# Bypass "Unsupported MimeType" error in VS Code Jupyter Notebooks when using Docker
save_chart(fig, "blog_event_engagement_efficiency", CHART_PREFIX)
plt.show()

- **The "Engagement Multiplier**  
Instead of plotting the raw numbers, plot the Ratio.
* **Metric**: **Ratio = Avg_Blog_Views / Avg_Event_Attendance**
* **Insight**: E.g.,"For every 1 person who attended a live event, X number of people read a blog post."

In [ ]:
# Calculate the Ratio (using a fresh variable name to be safe)
ratio_trend_df = annual_efficiency_summary_df.pivot(index='Year', columns='Metric_Type', values='Avg_Engagement')
ratio_trend_df['Engagement_Ratio'] = ratio_trend_df['Blog (Avg Views)'] / ratio_trend_df['Event (Avg Attendance)']
ratio_trend_df = ratio_trend_df.reset_index()

# --- Plotting ---
fig, ax = plt.subplots(figsize=(12, 7))

sns.lineplot(
    data=ratio_trend_df,
    x='Year', 
    y='Engagement_Ratio', 
    ax=ax, 
    color=CUSTOM_PALETTE['Blog Post'], 
    marker='o',        
    linewidth=2,       # Slightly thicker for visibility
    markersize=8,     
    label='Blog-to-Event Multiplier'
)

# Ensure ticks (both x and y) appear as per global style
ax.tick_params(axis='both', which='major', left=True, bottom=True)

ax.set_ylabel('Ratio (Views per 1 Attendee)', fontweight='bold')
ax.set_xlabel('Year', fontweight='bold')

for i in range(len(ratio_trend_df)):
    ax.annotate(f"{ratio_trend_df['Engagement_Ratio'].iloc[i]:.1f}x", 
                (ratio_trend_df['Year'].iloc[i], ratio_trend_df['Engagement_Ratio'].iloc[i]),
                textcoords="offset points", 
                xytext=(0, 15), 
                ha='center', 
                fontweight='bold',
                color=CUSTOM_PALETTE['Blog Post'])


fig.suptitle('The Engagement Multiplier: Growth Trend')

ax.set_title('"e.g.: For every 1 person who attended a live event in 2019, ~32 people read a blog post."', 
          pad=35)

# Legend (Centered between title and plot)
ax.legend(loc='lower center', bbox_to_anchor=(0.5, 1), frameon=False, ncol=1)

# Optional: Ensure the Y-axis starts at 0
ax.set_ylim(0, ratio_trend_df['Engagement_Ratio'].max() * 1.3)

# Bypass "Unsupported MimeType" error in VS Code Jupyter Notebooks when using Docker
save_chart(fig, "blog_event_engagement_efficiency_multiplier", CHART_PREFIX)
plt.show()